# 03 · Perfilamento da Bronze para validar o modelo

Testa, com consultas na tabela `mvp_reclamacoes.bronze.reclamacoes`, as hipóteses que sustentam o esquema estrela (fato de reclamações + `dim_tempo`, `dim_problema`, `dim_empresa`, `dim_local`, `dim_perfil`).

- **Recorte:** os segmentos "Bancos, Financeiras e Administradoras de Cartão" e "Empresas de Pagamento Eletrônico", nos 68 meses.
- **O notebook só lê,** não grava nada.
- **Saída:** cada célula imprime resultados agregados e pequenos com `print`, para que apareçam inteiros no VS Code e no workspace.
- **Espaços visíveis:** textos aparecem entre aspas (`repr`), para que espaços sobrando fiquem visíveis (ex.: `'N '`).

## 0. Parâmetros e funções de apoio

In [ ]:
from pyspark.sql import functions as F

TABELA = "mvp_reclamacoes.bronze.reclamacoes"
SEGMENTOS_RECORTE = [
    "Bancos, Financeiras e Administradoras de Cartão",
    "Empresas de Pagamento Eletrônico",
]

bronze = spark.table(TABELA)
recorte = bronze.filter(F.col("Segmento de Mercado").isin(SEGMENTOS_RECORTE))
mes = F.regexp_extract("_arquivo_origem", r"(\d{4}-\d{2})", 1)  # mês de referência, pelo nome do arquivo


def mostrar(df, n=100):
    """Imprime até n linhas de um DataFrame pequeno; textos com repr para mostrar espaços sobrando."""
    linhas = df.limit(n + 1).collect()
    print(" | ".join(df.columns))
    for linha in linhas[:n]:
        print(" | ".join("NULL" if v is None else repr(v) if isinstance(v, str) else str(v) for v in linha))
    if len(linhas) > n:
        print(f"... mais de {n} linhas, mostrando só as {n} primeiras")


def verificar_hierarquia(filho, pai):
    """Conta quantos valores de `filho` aparecem com mais de um valor de `pai` no recorte e lista os casos."""
    pares = recorte.groupBy(filho, pai).count()
    violacoes = pares.groupBy(filho).agg(F.count("*").alias("pais")).filter("pais > 1")
    print(f"{filho} → {pai}: {pares.select(filho).distinct().count()} valores de {filho!r}, "
          f"{violacoes.count()} com mais de um {pai!r}")
    mostrar(pares.join(violacoes, filho).orderBy(filho, F.desc("count")).drop("pais"), 40)

## H0. O filtro do recorte pega os segmentos certos?

Lista todos os segmentos com nomes parecidos, para confirmar que os dois do recorte estão escritos exatamente assim e que não existe variante com outra grafia.

In [ ]:
parecidos = bronze.filter(F.lower("Segmento de Mercado").rlike("banc|financ|cart|pagamento"))
mostrar(parecidos.groupBy("Segmento de Mercado").count().orderBy(F.desc("count")))
print(f"\nregistros no recorte: {recorte.count():,} de {bronze.count():,}")

## H1. A hierarquia do problema é consistente?

O modelo supõe uma `dim_problema` com `Área`, `Assunto`, `Grupo Problema` e `Problema`. Se um valor "filho" aparece com mais de um "pai", a chave da dimensão não pode ser só o filho.

In [ ]:
verificar_hierarquia("Assunto", "Área")
print()
verificar_hierarquia("Problema", "Grupo Problema")
print()
verificar_hierarquia("Problema", "Assunto")
print(f"\ncombinações distintas de Área/Assunto/Grupo Problema/Problema: "
      f"{recorte.select('Área', 'Assunto', 'Grupo Problema', 'Problema').distinct().count()}")

## H2. A taxonomia muda entre o início e o fim do período?

Compara os primeiros 12 meses (2021-01 a 2021-12) com os últimos 12 (2025-09 a 2026-08), que são as janelas da pergunta P2. Lista os valores que aparecem em só uma janela, com o primeiro e o último mês em que aparecem no período inteiro.

In [ ]:
janela = F.when(mes.between("2021-01", "2021-12"), "inicio").when(mes.between("2025-09", "2026-08"), "fim")
com_janela = recorte.withColumn("janela", janela).filter("janela IS NOT NULL")
for coluna in ["Grupo Problema", "Problema"]:
    por_janela = com_janela.groupBy(coluna).pivot("janela", ["inicio", "fim"]).count().fillna(0)
    periodo = recorte.groupBy(coluna).agg(F.min(mes).alias("primeiro_mes"), F.max(mes).alias("ultimo_mes"))
    so_uma = por_janela.filter("inicio = 0 OR fim = 0").join(periodo, coluna)
    print(f"== {coluna}: {por_janela.count()} valores nas duas janelas juntas, {so_uma.count()} em só uma delas")
    mostrar(so_uma.orderBy("primeiro_mes", coluna), 80)
    print()

## H3. Cada empresa tem um único segmento no período?

Considera todas as empresas que aparecem no recorte ao menos uma vez e olha todos os segmentos delas na Bronze inteira, porque uma empresa pode entrar ou sair do recorte ao mudar de segmento.

In [ ]:
empresas = recorte.select("Nome Fantasia").distinct()
segmentos = (
    bronze.join(empresas, "Nome Fantasia")
    .groupBy("Nome Fantasia", "Segmento de Mercado")
    .agg(F.count("*").alias("registros"), F.min(mes).alias("primeiro_mes"), F.max(mes).alias("ultimo_mes"))
)
varios = segmentos.groupBy("Nome Fantasia").count().filter("count > 1").select("Nome Fantasia")
print(f"empresas no recorte: {empresas.count()} | com mais de um segmento no período: {varios.count()}")
mostrar(segmentos.join(varios, "Nome Fantasia").orderBy("Nome Fantasia", "primeiro_mes"), 80)

## H4. A mesma empresa aparece com grafias diferentes?

Agrupa os nomes do recorte depois de uma normalização simples (minúsculas, sem espaços nas pontas, espaços repetidos reduzidos a um e sem acentos) e lista os grupos com mais de uma grafia original. Diferenças maiores, como "S.A." ou nomes antigos, não entram nesta checagem.

In [ ]:
COM_ACENTO = "áàâãäéèêëíìîïóòôõöúùûüçñ"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuucn"
normalizado = F.translate(F.regexp_replace(F.lower(F.trim("Nome Fantasia")), r"\s+", " "), COM_ACENTO, SEM_ACENTO)
grupos = (
    recorte.groupBy(normalizado.alias("nome_normalizado"))
    .agg(F.countDistinct("Nome Fantasia").alias("grafias"),
         F.sort_array(F.collect_set("Nome Fantasia")).alias("variantes"),
         F.count("*").alias("registros"))
    .filter("grafias > 1")
)
print(f"nomes distintos no recorte: {recorte.select('Nome Fantasia').distinct().count()} | "
      f"grupos com mais de uma grafia: {grupos.count()}")
mostrar(grupos.orderBy(F.desc("registros")), 60)
print(f"\nnomes com espaço nas pontas: "
      f"{recorte.filter(F.col('Nome Fantasia') != F.trim('Nome Fantasia')).select('Nome Fantasia').distinct().count()}")

## H5. Local: cada UF tem uma região? O mesmo nome de cidade aparece em mais de uma UF?

In [ ]:
mostrar(recorte.groupBy("Região").count().orderBy("Região"))
print()
verificar_hierarquia("UF", "Região")
print()
cidades = recorte.groupBy("Cidade").agg(F.countDistinct("UF").alias("ufs"), F.sort_array(F.collect_set("UF")).alias("lista_ufs"))
print(f"cidades distintas (só pelo nome): {cidades.count()} | pares UF + Cidade distintos: "
      f"{recorte.select('UF', 'Cidade').distinct().count()} | nomes de cidade em mais de uma UF: {cidades.filter('ufs > 1').count()}")
mostrar(cidades.filter("ufs > 1").orderBy(F.desc("ufs"), "Cidade"), 15)
print(f"\nnulos: Região={recorte.filter(F.col('Região').isNull()).count()} "
      f"UF={recorte.filter(F.col('UF').isNull()).count()} Cidade={recorte.filter(F.col('Cidade').isNull()).count()}")

## H6. Valores de `Sexo`, `Faixa Etária` e `Como Comprou Contratou` (candidatos da `dim_perfil`)

In [ ]:
for coluna in ["Sexo", "Faixa Etária", "Como Comprou Contratou"]:
    print(f"== {coluna}")
    mostrar(recorte.groupBy(coluna).count().orderBy(F.desc("count")))
    print()
print(f"combinações distintas Sexo × Faixa Etária × Como Comprou Contratou: "
      f"{recorte.select('Sexo', 'Faixa Etária', 'Como Comprou Contratou').distinct().count()}")

## H7. Domínios reais das colunas que viram medidas

Confirma as conversões do passo 2:
- `Respondida` S/N → booleano;
- `Avaliação Reclamação` → `foi_avaliada` e `foi_resolvida`;
- `Nota` e `Tempo Resposta` → inteiros.

Cruza também `Nota` com `Avaliação` e `Tempo Resposta` com `Respondida`, porque, pelo dicionário de dados do consumidor.gov.br, a satisfação só considera as reclamações avaliadas e o prazo só as respondidas.

In [ ]:
for coluna in ["Respondida", "Avaliação Reclamação", "Situação", "Nota do Consumidor"]:
    print(f"== {coluna}")
    mostrar(recorte.groupBy(coluna).count().orderBy(coluna))
    print()

print("== Avaliação Reclamação × nota preenchida")
mostrar(recorte.groupBy("Avaliação Reclamação", F.col("Nota do Consumidor").isNotNull().alias("tem_nota")).count()
        .orderBy("Avaliação Reclamação", "tem_nota"))
print("\n== Respondida × Tempo Resposta preenchido")
mostrar(recorte.groupBy("Respondida", F.col("Tempo Resposta").isNotNull().alias("tem_tempo")).count()
        .orderBy("Respondida", "tem_tempo"))

tempo = recorte.select(F.col("Tempo Resposta").alias("t"))
nao_numerico = tempo.filter(F.col("t").isNotNull() & ~F.col("t").rlike(r"^\d+$")).count()
numerico = tempo.filter(F.col("t").rlike(r"^\d+$")).select(F.col("t").cast("int").alias("t"))
resumo = numerico.agg(F.min("t"), F.max("t"), F.sum(F.when(F.col("t") > 10, 1).otherwise(0)).alias("acima_de_10")).first()
print(f"\n== Tempo Resposta: vazios={tempo.filter(F.col('t').isNull()).count():,} | não numéricos={nao_numerico} | "
      f"mín={resumo[0]} | máx={resumo[1]} | acima de 10 dias={resumo[2]:,}")

## H8. Formato e faixa de `Data Finalização` em cada arquivo

Confere o formato de todas as datas e se cada arquivo só tem datas do seu próprio mês.

In [ ]:
formato = (
    F.when(F.col("Data Finalização").isNull(), "nulo")
    .when(F.col("Data Finalização").rlike(r"^\d{4}-\d{2}-\d{2}$"), "aaaa-mm-dd")
    .when(F.col("Data Finalização").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/mm/aaaa")
    .otherwise("outro")
)
mostrar(bronze.groupBy(formato.alias("formato")).count())

por_arquivo = bronze.groupBy("_arquivo_origem").agg(
    F.min("Data Finalização").alias("menor"), F.max("Data Finalização").alias("maior")
).withColumn("mes_arquivo", F.regexp_extract("_arquivo_origem", r"(\d{4}-\d{2})", 1))
fora_do_mes = por_arquivo.filter((F.substring("menor", 1, 7) != F.col("mes_arquivo")) | (F.substring("maior", 1, 7) != F.col("mes_arquivo")))
print(f"\narquivos com datas fora do próprio mês: {fora_do_mes.count()} de {por_arquivo.count()}")
mostrar(fora_do_mes.orderBy("_arquivo_origem"))